# Contribute to the EU sovereign LLM from Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/individuum/dllm/blob/main/notebooks/colab_worker.ipynb)

Donate a Colab GPU session to the live distributed training run at [dllm.planetbass.de](https://dllm.planetbass.de).

**What this does:** runs the same DiLoCo worker that ships in the desktop client, here as a Jupyter cell. Connects outbound to the coord, downloads model state, runs one inner training loop per round, uploads the delta.

**Requirements:**
- Colab runtime with GPU enabled (Runtime → Change runtime type → T4 GPU). Free tier is enough for the 300M model.
- Python ≥3.10 (default on free Colab).
- ~5 min per round on T4. Free Colab sessions cap at ~12 h, so realistic per-session contribution is ~140 rounds.

**Note on identity:** each Colab kernel is a fresh VM. To preserve contribution-credit continuity across sessions, mount your Google Drive and persist the Ed25519 identity there — cell 2 does this for you.

## 1. Install the worker

In [ ]:
# Quick environment check.
import sys
print(f'Python {sys.version}')
assert sys.version_info >= (3, 10), 'Need Python ≥3.10. Change Colab runtime if you see this.'

# Use %pip (cell magic) not !pip — guarantees the install targets THIS kernel's Python.
# Drop -q so install errors are visible.
%pip install --upgrade git+https://github.com/individuum/dllm.git

# Sanity check: dllm + torch importable; CUDA visible.
import dllm, torch
print(f'✓ dllm at {dllm.__file__}')
assert torch.cuda.is_available(), 'No CUDA GPU detected. Runtime → Change runtime type → T4 GPU.'
print(f'✓ torch={torch.__version__} on {torch.cuda.get_device_name(0)} '
      f'({torch.cuda.get_device_properties(0).total_memory // (1024**3)} GB)')

## 2. Persist your contributor identity (recommended)

Mounting Google Drive lets your Ed25519 contributor key survive Colab session timeouts so the coord recognizes you across re-runs. Skip this cell to contribute anonymously each session (a new identity each time).

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    identity_dir = Path('/content/drive/MyDrive/dllm')
    identity_dir.mkdir(parents=True, exist_ok=True)
    os.environ['DLLM_IDENTITY_KEY'] = str(identity_dir / 'identity.key')
    print(f'✓ identity will persist at {os.environ["DLLM_IDENTITY_KEY"]}')
except Exception as e:
    print(f'Skipping drive mount: {e}\nYour contributor identity will reset each Colab session.')

## 3. Download a training shard

The 12 GB full training corpus would eat half your Colab session just to download. We ship a 500 MB sample slice via the coord's `/downloads/` endpoint that's enough to run real DiLoCo rounds and contribute back. (A future `/shard?worker_id=N` endpoint will stream-on-demand and skip this step entirely.)

In [ ]:
!mkdir -p /content/data/cache
!wget -q --show-progress https://dllm.planetbass.de/downloads/tokenizer.json -O /content/data/cache/tokenizer.json
!wget -q --show-progress https://dllm.planetbass.de/downloads/train-sample.bin -O /content/data/cache/train.bin
!wget -q --show-progress https://dllm.planetbass.de/downloads/val.bin   -O /content/data/cache/val.bin
!ls -lh /content/data/cache/

## 4. Start contributing

Runs the worker as a foreground process. Stop the cell (■ button) to disconnect cleanly — the coord auto-evicts your registration after inactivity timeout if you crash instead.

In [ ]:
# Pick your EU country (used for cohort attribution; not strictly enforced).
country = 'DE'   # @param {type:'string'}
# How many rounds to contribute this session. Each ~5 min on T4.
max_rounds = 60  # @param {type:'integer'}

!python -m dllm.client.worker \
    --coord https://dllm.planetbass.de \
    --preset 300M \
    --country {country} \
    --device cuda --require-gpu \
    --data /content/data/cache/train.bin \
    --val-data /content/data/cache/val.bin \
    --max-rounds {max_rounds}

## 5. Thank you!

Your contributed rounds appear at [dllm.planetbass.de](https://dllm.planetbass.de) under “active workers” while the session is live, and on the rounds chart afterwards.

Compute credits accumulate against your identity public key and translate into inference quota / governance vote weight / model-card recognition when the trained weights ship. See [PLAN.md §7.4](https://github.com/individuum/dllm/blob/main/PLAN.md#74-tier-d---individual-contributors) for the full recognition scheme.